# 01 – Data check: ECOSTRESS night coverage and a first MODIS vs ECOSTRESS comparison

This notebook answers three questions before any modelling:

1. How many ECOSTRESS night acquisitions cover central Istanbul in 2018–2025?
2. How many of them are actually cloud-free?
3. What does one hot summer night look like at 1 km (MODIS) and at 70 m (ECOSTRESS)?

Outputs: `data/ecostress_nights.csv` and `figures/01_modis_vs_ecostress.png`.

## Setup
On Colab this clones the repository and installs the requirements. Locally it uses the repository you are in.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/KeremBasimc/istanbul-night-heat-island"
if "google.colab" in sys.modules or Path("/content").exists():
    ROOT = Path("/content/istanbul-night-heat-island")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("Repository:", ROOT)

## Earthdata login

ECOSTRESS needs a free NASA Earthdata account. The password is never written in this notebook. Use one of these:

- **Colab (browser):** add `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` under *Secrets* (key icon on the left), or type them when prompted.
- **Colab CLI or local runs:** keep them in `~/.netrc` (`machine urs.earthdata.nasa.gov login ... password ...`).

In [ ]:
import earthaccess

try:  # Colab Secrets, if available
    from google.colab import userdata
    for key in ("EARTHDATA_USERNAME", "EARTHDATA_PASSWORD"):
        if key not in os.environ:
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                pass
except ImportError:
    pass

# Tries, in order: environment variables, ~/.netrc, then an interactive prompt.
auth = earthaccess.login()
print("Earthdata login ok:", auth.authenticated)

## 1. ECOSTRESS night acquisitions over central Istanbul

In [ ]:
import pandas as pd
from nhi import data

START, END = "2018-07-01", "2025-09-30"
granules = data.search_ecostress(START, END)
night = granules[granules.is_night].copy()
print(f"{len(granules)} granules, {len(night)} at night (21:00-05:00 local)")
night.groupby(night.local.dt.year).size().rename("night granules").to_frame().T

## 2. How many nights are cloud-free?
Downloads the `LST`, `cloud` and `water` layers. A land pixel counts as clear only if it has a valid LST value and the cloud mask marks it clear (the cloud mask is also 0 outside the swath). `clear_frac` is relative to the best-covered night.

In [ ]:
night = data.add_clear_fraction(night, DATA / "ecostress")
per_night = data.best_granule_per_night(night)   # one row per night: its clearest granule
per_night.to_csv(DATA / "ecostress_nights.csv", index=False)

summer = pd.to_datetime(per_night.night_date).dt.month.isin([6, 7, 8, 9])
for thr in (0.9, 0.7, 0.5):
    ok = per_night.clear_frac >= thr
    print(f"clear land >= {thr:.0%}: {ok.sum():3d} nights ({(ok & summer).sum()} in Jun-Sep)")

## 3. One hot summer night: MODIS vs ECOSTRESS
Picks the clearest summer night, takes the MODIS pass (Terra or Aqua) closest in time to the ECOSTRESS overpass, and plots both on one colour scale. MODIS is on the 980 m grid and ECOSTRESS on the 70 m grid; the two grids nest exactly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

coarse, fine = data.grids()
row = per_night[summer].sort_values("clear_frac", ascending=False).iloc[0]
print("Night of", row.night_date, "| ECOSTRESS local time", row.local, "| clear", round(row.clear_frac, 2))

paths = data.download([row.url_LST, row.url_cloud, row.url_water], DATA / "ecostress")
eco = data.ecostress_on_grid(*paths, fine)
passes = data.load_modis_night_passes(pd.Timestamp(row.night_date).date(), coarse)
platform, mp, gap = data.closest_modis_pass(passes, pd.Timestamp(row.local))
modis = mp["lst"]
print(f"MODIS {platform} pass at {mp['clock_h']:.1f} h local, {gap:.1f} h from ECOSTRESS")

both = np.concatenate([modis.values[np.isfinite(modis.values)], eco.values[np.isfinite(eco.values)]])
vmin, vmax = np.percentile(both, [2, 98])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
panels = [(axes[0], modis, f"MODIS {platform.title()} night LST, 1 km ({mp['clock_h']:.1f} h)"),
          (axes[1], eco, f"ECOSTRESS night LST, 70 m ({pd.Timestamp(row.local):%H:%M})")]
for ax, da, title in panels:
    im = da.plot.imshow(ax=ax, cmap="inferno", vmin=vmin, vmax=vmax, add_colorbar=False)
    ax.set_title(title); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")
fig.colorbar(im, ax=axes, label="Land surface temperature (°C)", shrink=0.85)
fig.suptitle(f"Central Istanbul, night of {row.night_date} (water and clouds masked)")
fig.savefig(FIGS / "01_modis_vs_ecostress.png", dpi=150)
plt.show()

## Next
- `02_benchmark_tsharp.ipynb`: TsHARP on the clear nights and its first error against ECOSTRESS.
- Fill in the access dates in `DATA_MANIFEST.md`.